# Uji internal MCMR (tanpa OpenMC)

> **Catatan penting tentang bug pelacakan.** Pada MCMR versi repo (commit `1636761`), geometri heterogen dapat memicu
> `RuntimeError: transport_one: exceeded 5000000 collisions for a single neutron`. Penyebab: setelah neutron melintasi batas sel
> interior sangat dekat tepi dunia, dorongan `1e-6` membuatnya berada di luar dunia; `Grid::distance_to_boundary` lalu
> menimpa `hit_side` menjadi `None` dan mengembalikan `1e-6`, sehingga neutron merayap keluar tanpa pernah memantul/bocor.
> Perbaikan minimal di akhir `Grid::distance_to_boundary` (`src/region.cpp`):
>
> ```cpp
> if (d_min <= 0.0) {
>     if (hit_side != Side::None) return 0.0;   // tepi dunia tetap diproses (pantul/bocor)
>     return 1e-6;                              // batas interior / lingkaran: langkah kecil
> }
> return d_min;
> ```
>
> Semua hasil yang dilaporkan di percakapan memakai build yang sudah ditambal. Tanpa tambalan, notebook ini bisa berhenti dengan error di atas
> (mis. varian "V4 cermin x", seed 2500).


# Uji internal MCMR (tanpa OpenMC)

Uji ini memeriksa geometri, kondisi batas, tally, dan estimator k **tanpa kode pembanding**.
Setiap uji punya jawaban yang diketahui dari dasar fisika atau matematika.

- **A1. Konsistensi tally:** neraca neutron eksak, tally fluks vs tally hitungan per sel, estimator k vs tally fluks, kebocoran sisi reflektif = 0.
- **A2. Invarian (metamorphic):** pemecahan sel, nama material kembar, lingkaran bermaterial kembar, skala, cermin, transpos, model setengah, simetri kebocoran.
- **A3. Statistik:** distribusi z-score di medium tak hingga (analitik), bias k vs N, rentang k, k awal, kasus tepi (12 grup acak).

Kriteria lulus: selisih statistik |z| < 3 untuk setiap perbandingan tunggal (dengan puluhan perbandingan, satu-dua nilai di 2 sampai 3 masih wajar).
Parameter ukuran run ada di sel konfigurasi (`QUICK=True` untuk uji cepat).

In [ ]:
import io, os, sys, time, json, contextlib
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import mcmr
print("mcmr dimuat dari:", mcmr.__file__)

QUICK = bool(int(os.environ.get("QUICK", "0")))   # True: ukuran kecil, hanya untuk memeriksa kode
OUT = "output_internal"; os.makedirs(OUT, exist_ok=True)

if QUICK:
    N_A1, NGEN_A1 = 4000, 30
    N_A2, NGEN_A2, R_A2 = 3000, 30, 3
    SKIP = 8
    M_Z, N_Z, NGEN_Z = 12, 2000, 25
    BUDGET, NS_BIAS = 2e5, [250, 1000]
    R_SCAN, N_SCAN, NGEN_SCAN = 2, 3000, 30
else:
    N_A1, NGEN_A1 = 20000, 60
    N_A2, NGEN_A2, R_A2 = 10000, 100, 8
    SKIP = 20
    M_Z, N_Z, NGEN_Z = 100, 5000, 60
    BUDGET, NS_BIAS = 3e6, [250, 500, 1000, 2500, 5000, 20000]
    R_SCAN, N_SCAN, NGEN_SCAN = 6, 10000, 100

SUMMARY = []   # (uji, ringkasan, lulus)
def catat(uji, ringkasan, lulus):
    SUMMARY.append((uji, ringkasan, bool(lulus)))
    print(f"  [{'LULUS' if lulus else 'GAGAL'}] {uji}: {ringkasan}")

## 0. Data material dan fungsi bantu
Dua grup. `C` = bahan bakar, `Fe` = reflektor, `Pb` = penyerap, `Be` = moderator. `U` adalah **kembaran** `C` (data identik, nama beda), dan `Cl`... tidak dipakai; kembaran reflektor dibuat dengan nama `Zr`.

In [ ]:
FUEL = dict(sigma_t=[0.474, 1.23], sigma_s=[0.45, 1.10], sigma_f=[0.004, 0.08], nu=[2.5, 2.43],
            chi_fission=[0.99, 0.01], chi_scatter=[[0.80, 0.20], [0.05, 0.95]])
REFL = dict(sigma_t=[0.40, 1.50], sigma_s=[0.38, 1.45], sigma_f=[0.0, 0.0], nu=[0.0, 0.0],
            chi_scatter=[[0.70, 0.30], [0.05, 0.95]])
ABSB = dict(sigma_t=[0.60, 1.40], sigma_s=[0.15, 0.20], sigma_f=[0.0, 0.0], nu=[0.0, 0.0],
            chi_scatter=[[0.50, 0.50], [0.10, 0.90]])
MODR = dict(sigma_t=[0.30, 0.90], sigma_s=[0.29, 0.89], sigma_f=[0.0, 0.0], nu=[0.0, 0.0],
            chi_scatter=[[0.60, 0.40], [0.10, 0.90]])

def make_lib(defs, alpha=1.0):
    """alpha: faktor skala panjang (semua Sigma dibagi alpha)."""
    lib = mcmr.MaterialLibrary()
    for name, d in defs.items():
        lib.add_material(name, [v / alpha for v in d["sigma_t"]], [v / alpha for v in d["sigma_s"]],
                         [v / alpha for v in d["sigma_f"]], d["nu"], d.get("chi_fission"), d.get("chi_scatter"))
    return lib

def make_world(xe, ye, mats, src, bc, circles=()):
    """xe, ye: batas sel (naik). mats/src: baris paling atas dulu. bc: dict top/bot/left/right."""
    w = mcmr.World(x_world=xe[-1], y_world=ye[-1], x_grid=list(xe[1:-1]), y_grid=list(ye[1:-1]),
                   material_matrix=mats, sources=src, bc_top=bc["top"], bc_bot=bc["bot"],
                   bc_left=bc["left"], bc_right=bc["right"])
    for (x, y, r, m, s) in circles:
        w.add_circle(x, y, r, m, source=s)
    return w

def jalankan(world, lib, N, n_gen, seed, res=None, k0=1.0):
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        sim = world.run(N=N, mode="group", group_materials=lib, n_gen=n_gen, max_save=0,
                        resolution=res, k=k0, seed=seed)
    return sim.get_tally()

def k_run(t, skip=SKIP):
    k = np.array(t.k_estimate)
    return k[skip:].mean()

def leak_total(t): return t.leak_left + t.leak_right + t.leak_bottom + t.leak_top

BC = lambda t, b, l, r: dict(top=t, bot=b, left=l, right=r)
VAC, REF = "vacuum", "reflective"

## A1. Konsistensi tally
Dunia 16×12 dengan 2×3 sel material (reflektor, bahan bakar, penyerap), batas campuran (kiri dan atas vakum, kanan dan bawah reflektif).
Dua versi: **W1** hanya grid (tally sel selaras dengan material, k bisa diperiksa) dan **W2** dengan dua lingkaran (`U` kembaran bahan bakar, `Be` moderator).

Uji:
1. **Neraca eksak:** tangkapan + fisi + kebocoran = N × jumlah generasi (setiap riwayat berakhir tepat satu kali).
2. **Sisi reflektif tidak bocor** (kebocoran = 0 persis).
3. **Tally fluks vs tally hitungan per sel:** nilai harapan tangkapan = Σ_g Σc,g · T_g dan fisi = Σ_g Σf,g · T_g (T = panjang lintasan). Selisih ternormalisasi z = (obs − prediksi)/√prediksi harus ~ N(0,1).
4. **Estimator k vs tally fluks (W1):** N·Σ_gen k_gen sama dengan total neutron fisi yang diproduksi (Σ ν_g Σf,g T_g).

In [ ]:
A1_DEFS = {"C": FUEL, "Fe": REFL, "Pb": ABSB, "U": FUEL, "Be": MODR}
A1_XE, A1_YE = [0, 5, 10, 16], [0, 6, 12]
A1_MATS = [["Fe", "C", "Pb"], ["C", "C", "Fe"]]
A1_SRC = [[0, 1, 0], [1, 1, 0]]
A1_BC = BC(VAC, REF, VAC, REF)
A1_CIRC = [(7.5, 3.0, 1.5, "U", 1), (13.0, 9.0, 2.0, "Be", 0)]
A1_RES = (32, 24)          # sel tally 0.5 x 0.5, selaras dengan batas material

def mat_di(x, y, circles):
    for (cx, cy, r, m, s) in reversed(circles):          # lingkaran terakhir di atas
        if (x - cx) ** 2 + (y - cy) ** 2 <= r * r: return m
    ix = np.searchsorted(A1_XE, x) - 1; iy = np.searchsorted(A1_YE, y) - 1
    return A1_MATS[len(A1_YE) - 2 - iy][ix]

def a1_satu_run(circles, seed, cek_k):
    w = make_world(A1_XE, A1_YE, A1_MATS, A1_SRC, A1_BC, circles)
    t = jalankan(w, make_lib(A1_DEFS), N_A1, NGEN_A1, seed, res=A1_RES)
    nx, ny, G = t.tally_nx, t.tally_ny, t.flux_groups
    flux = np.array(t.flux_tally).reshape(G, ny, nx)
    fis = np.array(t.fission_tally).reshape(ny, nx); cap = np.array(t.absorp_tally).reshape(ny, nx)
    n_gen_done = len(t.k_estimate)
    hasil = {}
    # 1. neraca eksak
    hasil["neraca"] = (int(cap.sum() + fis.sum() + leak_total(t)), N_A1 * n_gen_done)
    # 2. sisi reflektif
    hasil["bocor_reflektif"] = (t.leak_right, t.leak_bottom)
    hasil["bocor_vakum"] = (t.leak_left, t.leak_top)
    # klasifikasi sel tally
    dx, dy = t.tally_dx, t.tally_dy
    murni = np.full((ny, nx), None, dtype=object)
    for iy in range(ny):
        for ix in range(nx):
            cx, cy = (ix + 0.5) * dx, (iy + 0.5) * dy
            if any(abs(np.hypot(cx - c[0], cy - c[1]) - c[2]) < 0.5 * np.hypot(dx, dy) for c in circles):
                continue
            murni[iy, ix] = mat_di(cx, cy, circles)
    cap_p = np.zeros((ny, nx)); fis_p = np.zeros((ny, nx)); nuf_p = np.zeros((ny, nx)); ok = np.zeros((ny, nx), bool)
    for iy in range(ny):
        for ix in range(nx):
            m = murni[iy, ix]
            if m is None: continue
            d = A1_DEFS[m]; ok[iy, ix] = True
            sc = np.array(d["sigma_t"]) - np.array(d["sigma_s"]) - np.array(d["sigma_f"])
            cap_p[iy, ix] = (sc * flux[:, iy, ix]).sum()
            fis_p[iy, ix] = (np.array(d["sigma_f"]) * flux[:, iy, ix]).sum()
            nuf_p[iy, ix] = (np.array(d["nu"]) * np.array(d["sigma_f"]) * flux[:, iy, ix]).sum()
    for nama, obs, pred in (("tangkapan", cap, cap_p), ("fisi", fis, fis_p)):
        sel = ok & (pred >= 10)
        z = (obs[sel] - pred[sel]) / np.sqrt(pred[sel])
        hasil[nama] = dict(n=int(sel.sum()), mean_z=float(z.mean()), std_z=float(z.std(ddof=1)),
                           max_abs_z=float(np.abs(z).max()), rasio_total=float(obs[ok].sum() / pred[ok].sum()),
                           chi2_dof=float((z ** 2).mean()))
    if cek_k:
        prod_real = N_A1 * np.array(t.k_estimate).sum()
        prod_pred = nuf_p[ok].sum()
        F = fis.sum(); nu_bar = 2.47
        sd = np.sqrt(F * nu_bar ** 2 + 0.25 * F)
        hasil["k_vs_fluks"] = (float(prod_real), float(prod_pred), float((prod_real - prod_pred) / sd))
    return hasil

In [ ]:
print("=== A1. Konsistensi tally ===")
for label, circ, cek_k in (("W1 (grid)", [], True), ("W2 (grid + 2 lingkaran)", A1_CIRC, False)):
    print(f"\n-- {label}: N={N_A1}, generasi={NGEN_A1}, 3 seed --")
    H = [a1_satu_run(circ, 100 + i, cek_k) for i in range(3)]
    catat(f"A1 neraca eksak [{label}]", "obs == N*gen pada semua seed: " + str([h['neraca'] for h in H]),
          all(h["neraca"][0] == h["neraca"][1] for h in H))
    catat(f"A1 sisi reflektif tidak bocor [{label}]",
          f"kanan/bawah = {[h['bocor_reflektif'] for h in H]}; sisi vakum bocor = {[h['bocor_vakum'] for h in H]}",
          all(h["bocor_reflektif"] == (0, 0) for h in H) and all(min(h["bocor_vakum"]) > 0 for h in H))
    for nama in ("tangkapan", "fisi"):
        zs = [h[nama] for h in H]
        ok = all(abs(z["mean_z"]) < 0.5 and 0.8 < z["std_z"] < 1.25 and z["max_abs_z"] < 4.5 and abs(z["rasio_total"] - 1) < 0.01 for z in zs)
        catat(f"A1 tally fluks vs hitungan: {nama} [{label}]",
              "per seed (n sel, mean z, std z, max|z|, rasio total) = " +
              "; ".join(f"({z['n']}, {z['mean_z']:+.2f}, {z['std_z']:.2f}, {z['max_abs_z']:.1f}, {z['rasio_total']:.4f})" for z in zs), ok)
    if cek_k:
        ks = [h["k_vs_fluks"] for h in H]
        catat(f"A1 estimator k vs produksi dari tally fluks [{label}]",
              "z per seed = " + ", ".join(f"{v[2]:+.2f}" for v in ks) +
              "; rasio = " + ", ".join(f"{v[0]/v[1]:.5f}" for v in ks), all(abs(v[2]) < 3 for v in ks))

## A2. Uji invarian
Soal dasar **B**: reflektor (4 cm), bahan bakar (8 cm), reflektor (8 cm); kiri dan kanan vakum, atas dan bawah reflektif.
Setiap varian adalah **soal fisik yang sama** dengan representasi berbeda, sehingga k harus sama dalam batas statistik.
Perbandingan memakai z = (k_varian − k_B)/√(se² + se²) dari sebaran antar run.

In [ ]:
A2_DEFS = {"C": FUEL, "U": FUEL, "Fe": REFL, "Zr": REFL}
SRC_ROW = lambda mats: [[1 if m in ("C", "U") else 0 for m in row] for row in mats]

def varian(nama):
    """return (world, lib) untuk varian soal B."""
    lib1 = make_lib(A2_DEFS)
    base_mats = [["Fe", "C", "Fe"]]
    base_xe, base_ye = [0, 4, 12, 20], [0, 10]
    if nama == "B (dasar)":
        return make_world(base_xe, base_ye, base_mats, SRC_ROW(base_mats), BC(REF, REF, VAC, VAC)), lib1
    if nama == "V1 sel dipecah + nama kembaran":
        xe, ye = [0, 2, 4, 8, 10, 12, 16, 20], [0, 5, 10]
        top = ["Fe", "Zr", "C", "U", "C", "Zr", "Fe"]; bot = ["Zr", "Fe", "U", "C", "U", "Fe", "Zr"]
        return make_world(xe, ye, [top, bot], SRC_ROW([top, bot]), BC(REF, REF, VAC, VAC)), lib1
    if nama == "V2 lingkaran bermaterial kembar":
        w = make_world(base_xe, base_ye, base_mats, SRC_ROW(base_mats), BC(REF, REF, VAC, VAC),
                       circles=[(8, 5, 3.0, "U", 0), (16, 5, 3.0, "Zr", 0), (9, 6, 1.5, "U", 0), (2, 5, 1.8, "Zr", 0)])
        return w, lib1
    if nama.startswith("V3 skala"):
        a = 2.0 if "x2" in nama else 0.5
        return (make_world([v * a for v in base_xe], [v * a for v in base_ye], base_mats, SRC_ROW(base_mats),
                           BC(REF, REF, VAC, VAC)), make_lib(A2_DEFS, alpha=a))
    if nama == "V4 cermin x":
        xe = [0, 8, 16, 20]
        return make_world(xe, base_ye, base_mats, SRC_ROW(base_mats), BC(REF, REF, VAC, VAC)), lib1
    if nama == "V5 transpos x<->y":
        mats = [["Fe"], ["C"], ["Fe"]]      # baris paling atas = y terbesar
        return make_world([0, 10], [0, 4, 12, 20], [["Fe"], ["C"], ["Fe"]], SRC_ROW(mats), BC(VAC, VAC, REF, REF)), lib1
    raise ValueError(nama)

NAMA_V = ["B (dasar)", "V1 sel dipecah + nama kembaran", "V2 lingkaran bermaterial kembar",
          "V3 skala x2", "V3 skala x0.5", "V4 cermin x", "V5 transpos x<->y"]

print("\n=== A2. Uji invarian ===")
print(f"N={N_A2}, generasi={NGEN_A2}, buang {SKIP}, {R_A2} run per varian")
K2 = {}
for iv, nama in enumerate(NAMA_V):
    w, lib = varian(nama)
    t0 = time.time()
    ks = np.array([k_run(jalankan(w, lib, N_A2, NGEN_A2, 2000 + 100 * iv + r)) for r in range(R_A2)])
    K2[nama] = (ks.mean(), ks.std(ddof=1) / np.sqrt(R_A2))
    print(f"  {nama:34s} k = {K2[nama][0]:.5f} +- {K2[nama][1]:.5f}   ({time.time()-t0:.0f} s)")
kB, eB = K2["B (dasar)"]
for nama in NAMA_V[1:]:
    k, e = K2[nama]; z = (k - kB) / np.hypot(e, eB)
    catat(f"A2 {nama}", f"k = {k:.5f} vs dasar {kB:.5f}, selisih {1e5*(k-kB):+.0f} pcm, z = {z:+.2f}", abs(z) < 3)

### A2 lanjutan: model setengah dan simetri kebocoran
**Model setengah:** soal simetris penuh (reflektor 5, bahan bakar 10, reflektor 5; vakum kiri dan kanan) vs setengahnya (bahan bakar 5, reflektor 5; kiri **reflektif** sebagai bidang simetri, kanan vakum).
**Simetri kebocoran:** pada soal simetris penuh, kebocoran kiri harus sama dengan kanan (dan atas dengan bawah jika atas/bawah vakum), diuji dengan z = (L − R)/√(L + R).

In [ ]:
full_mats = [["Fe", "C", "Fe"]]
w_full = make_world([0, 5, 15, 20], [0, 10], full_mats, SRC_ROW(full_mats), BC(REF, REF, VAC, VAC))
half_mats = [["C", "Fe"]]
w_half = make_world([0, 5, 10], [0, 10], half_mats, SRC_ROW(half_mats), BC(REF, REF, REF, VAC))
lib = make_lib(A2_DEFS)
kf = np.array([k_run(jalankan(w_full, lib, N_A2, NGEN_A2, 5000 + r)) for r in range(R_A2)])
kh = np.array([k_run(jalankan(w_half, lib, N_A2, NGEN_A2, 5100 + r)) for r in range(R_A2)])
ef, eh = kf.std(ddof=1) / np.sqrt(R_A2), kh.std(ddof=1) / np.sqrt(R_A2)
z = (kh.mean() - kf.mean()) / np.hypot(ef, eh)
print(f"\n  model penuh  k = {kf.mean():.5f} +- {ef:.5f}\n  model setengah k = {kh.mean():.5f} +- {eh:.5f}")
catat("A2 model setengah (batas reflektif) vs model penuh", f"selisih {1e5*(kh.mean()-kf.mean()):+.0f} pcm, z = {z:+.2f}", abs(z) < 3)

# simetri kebocoran: semua vakum, soal simetris di x dan y
w_sym = make_world([0, 5, 15, 20], [0, 10], full_mats, SRC_ROW(full_mats), BC(VAC, VAC, VAC, VAC))
Lc = Rc = Tc = Bc = 0
for r in range(R_A2):
    t = jalankan(w_sym, lib, N_A2, NGEN_A2, 5200 + r)
    Lc += t.leak_left; Rc += t.leak_right; Tc += t.leak_top; Bc += t.leak_bottom
zLR = (Lc - Rc) / np.sqrt(Lc + Rc); zTB = (Tc - Bc) / np.sqrt(Tc + Bc)
catat("A2 simetri kebocoran kiri vs kanan", f"L = {Lc}, R = {Rc}, z = {zLR:+.2f}", abs(zLR) < 3)
catat("A2 simetri kebocoran atas vs bawah", f"T = {Tc}, B = {Bc}, z = {zTB:+.2f}", abs(zTB) < 3)

## A3. Validasi statistik (medium tak hingga, jawaban analitik)
Fungsi analitik: `A = diag(Σt) − S`, `S[tujuan, asal] = Σs,asal · P[asal][tujuan]`, `k = (νΣf)ᵀ A⁻¹ χ`, spektrum tally `∝ A⁻¹u + (n_gen − 1) A⁻¹χ` (generasi 1 dari sumber seragam u).

In [ ]:
def analitik(c, n_gen=None):
    st = np.array(c["sigma_t"], float); ss = np.array(c["sigma_s"], float)
    nsf = np.array(c["nu"], float) * np.array(c["sigma_f"], float)
    P = np.array(c["chi_scatter"], float); sums = P.sum(1, keepdims=True)
    P = np.divide(P, sums, out=np.zeros_like(P), where=sums > 0)
    A = np.diag(st) - (ss[:, None] * P).T
    G = len(st)
    if c.get("chi_fission") is not None:
        chi = np.array(c["chi_fission"], float); chi = chi / chi.sum()
        x = np.linalg.solve(A, chi); k = nsf @ x
        out = dict(k=k)
        if n_gen:
            tl = np.linalg.solve(A, np.ones(G) / G) + (n_gen - 1) * x; out["phi_tally"] = tl / tl.sum()
        return out
    w, v = np.linalg.eig(np.linalg.solve(A, np.diag(nsf)))      # anak fisi mewarisi grup induk
    i = np.argmax(w.real); return dict(k=float(w[i].real))

def inf_world(L=10.0, mat="C"):
    return make_world([0, L], [0, L], [[mat]], [[1]], BC(REF, REF, REF, REF))

### A3a. Distribusi z-score di banyak seed
Jika estimator k tak bias **dan** galat yang dilaporkan jujur, maka z_i = (k̄_i − k_analitik)/se_i harus berdistribusi N(0,1): rata-rata 0, deviasi 1, ~68% dalam 1σ, ~95% dalam 2σ.
`se_i` dihitung secara naif dari sebaran antar generasi dalam satu run (menganggap generasi independen), sehingga uji ini juga memeriksa dampak korelasi antar generasi.

In [ ]:
print("\n=== A3a. Distribusi z-score ===")
ka = analitik(FUEL)["k"]
wz, lz = inf_world(), make_lib({"C": FUEL})
kb, se, ac = [], [], []
for i in range(M_Z):
    k = np.array(jalankan(wz, lz, N_Z, NGEN_Z, 9000 + i).k_estimate)[SKIP // 2:]
    kb.append(k.mean()); se.append(k.std(ddof=1) / np.sqrt(len(k)))
    d = k - k.mean(); ac.append((d[:-1] * d[1:]).sum() / (d * d).sum())
kb, se, ac = map(np.array, (kb, se, ac)); zz = (kb - ka) / se
print(f"  k analitik = {ka:.5f}; M = {M_Z} run, N = {N_Z}, {NGEN_Z - SKIP // 2} generasi dipakai per run")
print(f"  rata-rata (k_run - k_an) = {1e5*(kb-ka).mean():+.0f} pcm +- {1e5*kb.std(ddof=1)/np.sqrt(M_Z):.0f} pcm")
print(f"  z: mean = {zz.mean():+.3f} (harapan 0 +- {1/np.sqrt(M_Z):.3f}), std = {zz.std(ddof=1):.3f} (harapan 1), "
      f"dalam 1 sigma = {np.mean(np.abs(zz) < 1):.2f} (0.68), dalam 2 sigma = {np.mean(np.abs(zz) < 2):.2f} (0.95)")
print(f"  rasio sebaran nyata/se naif = {kb.std(ddof=1)/se.mean():.3f}; autokorelasi lag-1 rata-rata = {ac.mean():+.3f}")
catat("A3a z-score: rata-rata 0", f"mean z = {zz.mean():+.3f} +- {1/np.sqrt(M_Z):.3f}", abs(zz.mean()) < 3 / np.sqrt(M_Z))
catat("A3a z-score: deviasi 1 (galat jujur)", f"std z = {zz.std(ddof=1):.3f}", 0.75 < zz.std(ddof=1) < 1.30)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
ax[0].hist(zz, bins=15, density=True, alpha=.7, label="z hasil"); xs = np.linspace(-4, 4, 200)
ax[0].plot(xs, np.exp(-xs**2 / 2) / np.sqrt(2 * np.pi), "k--", label="N(0,1)"); ax[0].set_xlabel("z"); ax[0].legend()
ax[0].set_title("Distribusi z-score k (medium tak hingga)")
srt = np.sort(zz); q = np.linspace(0.5 / M_Z, 1 - 0.5 / M_Z, M_Z)
from math import erf
inv = np.array([np.sqrt(2) * np.sqrt(np.pi) / 2 * 0 for _ in q])
try:
    from scipy.stats import norm; th = norm.ppf(q)
except Exception:
    th = np.sort(np.random.default_rng(0).standard_normal(M_Z))
ax[1].plot(th, srt, "o", ms=3); lim = [-3.5, 3.5]; ax[1].plot(lim, lim, "k--"); ax[1].set_xlabel("kuantil N(0,1)"); ax[1].set_ylabel("kuantil z")
ax[1].set_title("QQ plot"); plt.tight_layout(); plt.savefig(f"{OUT}/a3a_zscore.png", dpi=110); plt.show()

### A3b. Bias k terhadap N
Estimator MCMR memakai umpan balik k dan resampling bank dengan pengembalian, sehingga bias pada N kecil perlu dicek. Total riwayat dibuat sama untuk setiap N (`BUDGET`), jadi galat statistik seragam.
- **Medium tak hingga:** bias dibandingkan dengan k analitik.
- **Soal bocor B (reflektor-bahan bakar-reflektor):** bias dibandingkan dengan nilai N besar.

In [ ]:
print("\n=== A3b. Bias k vs N ===")
def k_banyak_run(world, lib, N, budget, ngen=50, skip=10, seed0=0):
    R = max(2, int(budget / (N * ngen)))
    ks = []
    for r in range(R):
        k = np.array(jalankan(world, lib, N, ngen, seed0 + r).k_estimate)[skip:]
        ks.append(k.mean())
    ks = np.array(ks); return ks.mean(), ks.std(ddof=1) / np.sqrt(R), R

res_inf = []
for N in NS_BIAS:
    m, e, R = k_banyak_run(wz, lz, N, BUDGET, seed0=20000 + N)
    res_inf.append((N, m, e)); print(f"  tak hingga N={N:6d} (R={R:4d}): k - k_an = {1e5*(m-ka):+7.0f} +- {1e5*e:.0f} pcm")
wB, lB = varian("B (dasar)")
res_B = []
for N in NS_BIAS:
    m, e, R = k_banyak_run(wB, lB, N, BUDGET, seed0=40000 + N)
    res_B.append((N, m, e)); print(f"  soal bocor B N={N:6d} (R={R:4d}): k = {m:.5f} +- {e:.5f}")
Nr, mr, er = res_B[-1]
for (N, m, e) in res_inf:
    z = (m - ka) / e
    catat(f"A3b bias tak hingga, N={N}", f"{1e5*(m-ka):+.0f} +- {1e5*e:.0f} pcm, z = {z:+.2f}", abs(z) < 3)
for (N, m, e) in res_B[:-1]:
    z = (m - mr) / np.hypot(e, er)
    catat(f"A3b bias soal bocor B, N={N} vs N={Nr}", f"{1e5*(m-mr):+.0f} pcm, z = {z:+.2f}", abs(z) < 3)
fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.errorbar([r[0] for r in res_inf], [1e5 * (r[1] - ka) for r in res_inf], [1e5 * r[2] for r in res_inf], fmt="o-", label="tak hingga (vs analitik)")
ax.errorbar([r[0] for r in res_B], [1e5 * (r[1] - mr) for r in res_B], [1e5 * np.hypot(r[2], er) for r in res_B], fmt="s-", label=f"soal bocor B (vs N={Nr})")
ax.axhline(0, color="k", ls="--"); ax.set_xscale("log"); ax.set_xlabel("N (partikel per generasi)"); ax.set_ylabel("bias k (pcm)"); ax.legend()
plt.tight_layout(); plt.savefig(f"{OUT}/a3b_bias_vs_N.png", dpi=110); plt.show()

### A3c. Rentang k dan k awal
`νΣf` diskalakan sehingga k∞ berkisar dari sangat subkritis sampai superkritis, termasuk tepat kritis. Lalu `k_initial` dibuat jauh dari k sebenarnya.

In [ ]:
print("\n=== A3c. Rentang k dan k awal ===")
for s in (0.35, 0.8, 0.95, 1.0 / ka, 1.05, 1.5, 2.5):
    c = dict(FUEL, nu=[v * s for v in FUEL["nu"]]); kan = analitik(c)["k"]
    lib_s = make_lib({"C": c}); ks = np.array([k_run(jalankan(wz, lib_s, N_SCAN, NGEN_SCAN, 60000 + int(1000 * s) + r)) for r in range(R_SCAN)])
    m, e = ks.mean(), ks.std(ddof=1) / np.sqrt(R_SCAN); z = (m - kan) / e
    catat(f"A3c k_analitik = {kan:.4f}", f"k = {m:.5f} +- {e:.5f}, {1e5*(m-kan):+.0f} pcm, z = {z:+.2f}", abs(z) < 3)
for k0 in (0.3, 3.0):
    ks = np.array([k_run(jalankan(wz, lz, N_SCAN, NGEN_SCAN, 70000 + r, k0=k0)) for r in range(R_SCAN)])
    m, e = ks.mean(), ks.std(ddof=1) / np.sqrt(R_SCAN); z = (m - ka) / e
    catat(f"A3c k_initial = {k0}", f"k = {m:.5f} +- {e:.5f}, {1e5*(m-ka):+.0f} pcm, z = {z:+.2f}", abs(z) < 3)

### A3d. Kasus tepi: 12 grup acak
Data acak (seed tetap) dengan matriks hamburan jarang, satu grup tanpa hamburan (baris nol, `sigma_s = 0`), satu grup **terisolasi** (hanya hamburan ke dirinya dan tidak diisi grup lain, χ = 0), dan χ jarang. Dibandingkan: k dan spektrum fluks tally (analitik eksak, termasuk generasi 1).
Ditambah satu kasus tanpa `chi_fission` (anak fisi mewarisi grup induk).

In [ ]:
print("\n=== A3d. Kasus tepi ===")
rng = np.random.default_rng(7); G = 12
ss = rng.uniform(0.2, 0.8, G); cap = rng.uniform(0.01, 0.10, G); sf = rng.uniform(0.0, 0.06, G) * (rng.random(G) < 0.6)
P = rng.random((G, G)) * (rng.random((G, G)) < 0.5)
ss[5] = 0.0; P[5, :] = 0.0                      # grup 5: tidak pernah menghambur
P[:, 8] = 0.0; P[8, :] = 0.0; P[8, 8] = 1.0     # grup 8: terisolasi
for g in range(G):
    if ss[g] > 0 and P[g].sum() == 0: P[g, (g + 1) % G if (g + 1) % G != 8 else 0] = 1.0
chi = rng.random(G) * (rng.random(G) < 0.5); chi[8] = 0.0; chi[0] += 0.3
sf[8] = 0.0
nu = rng.uniform(2.3, 2.9, G)
st = ss + cap + sf
c12 = dict(sigma_t=st.tolist(), sigma_s=ss.tolist(), sigma_f=sf.tolist(), nu=nu.tolist(),
           chi_fission=chi.tolist(), chi_scatter=P.tolist())
k12 = analitik(c12, NGEN_SCAN)
print(f"  12G: k analitik = {k12['k']:.5f}")
lib12 = make_lib({"C": c12}); runs = []
for r in range(R_SCAN):
    t = jalankan(wz, lib12, N_SCAN, NGEN_SCAN, 80000 + r)
    fl = np.array(t.flux_tally).reshape(t.flux_groups, -1).sum(1); runs.append((k_run(t), fl / fl.sum()))
ks = np.array([r[0] for r in runs]); m, e = ks.mean(), ks.std(ddof=1) / np.sqrt(R_SCAN); z = (m - k12["k"]) / e
catat("A3d 12 grup acak: k", f"k = {m:.5f} +- {e:.5f} vs {k12['k']:.5f}, z = {z:+.2f}", abs(z) < 3)
fm = np.array([r[1] for r in runs]); em = fm.std(0, ddof=1) / np.sqrt(R_SCAN)
zs = (fm.mean(0) - k12["phi_tally"]) / np.maximum(em, 1e-12)
ok = np.abs(zs[em > 0]).max() < 4.5 and abs(fm.mean(0)[8] - k12["phi_tally"][8]) < 3e-3 * k12["phi_tally"][8] + 1e-6
catat("A3d 12 grup acak: spektrum fluks (12 grup)",
      f"max|z| = {np.abs(zs).max():.2f}; grup 8 (terisolasi) = {fm.mean(0)[8]:.5f} vs {k12['phi_tally'][8]:.5f}; "
      f"grup 5 (tanpa hamburan) = {fm.mean(0)[5]:.5f} vs {k12['phi_tally'][5]:.5f}", ok)

c3 = dict(sigma_t=[0.423, 0.645, 1.138], sigma_s=[0.40, 0.60, 1.00], sigma_f=[0.003, 0.015, 0.078], nu=[2.8, 2.6, 2.43],
          chi_fission=None, chi_scatter=[[0.60, 0.35, 0.05], [0.02, 0.70, 0.28], [0.00, 0.08, 0.92]])
k3 = analitik(c3)["k"]; lib3 = make_lib({"C": c3})
ks = np.array([k_run(jalankan(wz, lib3, N_SCAN, NGEN_SCAN, 90000 + r)) for r in range(R_SCAN)])
m, e = ks.mean(), ks.std(ddof=1) / np.sqrt(R_SCAN); z = (m - k3) / e
catat("A3d tanpa chi_fission (anak mewarisi grup induk)", f"k = {m:.5f} +- {e:.5f} vs analitik {k3:.5f}, z = {z:+.2f}", abs(z) < 3)

### A3e. k_initial: apakah ada transien?
Tindak lanjut dari hasil borderline k_initial = 0,3 (−64 pcm, z = −3,4 dengan 30 run). Banyak run pendek di medium tak hingga; k rata-rata dihitung dengan `n_skip` yang divariasikan.
Jika selisih berasal dari transien generasi awal, ia menyusut saat `n_skip` membesar; jika berasal dari bias tetap, ia tidak berubah.

In [ ]:
print("\n=== A3e. k_initial dan transien ===")
R_E, N_E, NG_E = (150, 5000, 60) if not QUICK else (4, 2000, 25)
arrs = {}
for k0 in (0.3, 1.0, 3.0):
    arrs[k0] = np.array([np.array(jalankan(wz, lz, N_E, NG_E, 400000 + int(10 * k0) * 1000 + r, k0=k0).k_estimate) for r in range(R_E)])
    print(f"  k_initial = {k0}: {R_E} run selesai")
print(f"  k analitik = {ka:.5f}; deviasi k rata-rata per generasi 1..6 (pcm): ")
for k0, A in arrs.items():
    print(f"    k0={k0}: " + ", ".join(f"{1e5*(A[:, g].mean()-ka):+.0f}" for g in range(6)))
print("  selisih k rata-rata (pcm +- se) menurut n_skip:")
for k0, A in arrs.items():
    out = []
    for sk in (1, 3, 5, 10, 20):
        pr = A[:, sk:].mean(axis=1); m, e = pr.mean(), pr.std(ddof=1) / np.sqrt(len(pr))
        out.append(f"skip={sk}: {1e5*(m-ka):+.0f}+-{1e5*e:.0f} (z={(m-ka)/e:+.1f})")
    print(f"    k0={k0}: " + "; ".join(out))
for k0, A in arrs.items():
    pr = A[:, 10:].mean(axis=1); m, e = pr.mean(), pr.std(ddof=1) / np.sqrt(len(pr)); z = (m - ka) / e
    catat(f"A3e k_initial = {k0} (skip=10, {R_E} run)", f"{1e5*(m-ka):+.0f} +- {1e5*e:.0f} pcm, z = {z:+.2f}", abs(z) < 3)

## Ringkasan

In [ ]:
print("\n" + "=" * 100 + "\nRINGKASAN UJI INTERNAL\n" + "=" * 100)
for uji, ring, ok in SUMMARY:
    print(f"[{'LULUS' if ok else 'GAGAL'}] {uji}")
n_ok = sum(1 for s in SUMMARY if s[2])
print(f"\nTotal: {n_ok}/{len(SUMMARY)} lulus")
json.dump([dict(uji=u, ringkasan=r, lulus=o) for u, r, o in SUMMARY], open(f"{OUT}/ringkasan_uji_internal.json", "w"), indent=1, ensure_ascii=False)